# Notebook 05 — Live Demo App (Kaggle T4 + ngrok)

Launches the Gradio dual-mode CXR app on Kaggle GPU with a public ngrok URL.

## Required Inputs (right sidebar → + Add Input):
1. **Kaggle Dataset**: `raddar/chest-xrays-indiana-university` (the images)
2. **Notebook output**: Your `01+02 - Data + QA + Indexes Complete` (has the indexes)

## Required Secrets (Add-ons → Secrets):
- `HF_TOKEN` — HuggingFace token with MedGemma access
- `NGROK_TOKEN` — Free from https://dashboard.ngrok.com

## Kaggle Settings:
- **Accelerator**: GPU T4 x2 (or any GPU)
- **Internet**: ON

## Step 1: Fix huggingface_hub version (must run first)

Kaggle has a partial install. Reinstall clean, then restart kernel.

In [ ]:
!pip install -q --upgrade --force-reinstall huggingface_hub
print('✓ huggingface_hub reinstalled')
print('⚠️  NOW RESTART KERNEL: Run menu → Restart Kernel')
print('   Then run cells from Step 2 onwards.')

## Step 2: Install all dependencies

In [ ]:
!pip install -q --upgrade peft transformers
!pip install -q gradio pyngrok colpali-engine accelerate bitsandbytes
!pip install -q open-clip-torch faiss-cpu
!pip install -q --upgrade torchao
print('✓ All packages installed')

## Step 3: Configure paths (EDIT THESE TO MATCH YOUR INPUTS)

In [ ]:
import os, sys, glob

# Edit these paths to match your Kaggle Input mounts
CORPUS_PATH       = '/kaggle/input/reports-corpus/reports_corpus.csv'
COLPALI_INDEX_DIR = '/kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete-v2/colpali_index'
CLIP_INDEX_DIR    = '/kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete-v2/clip_index'

# Verify
for name, path in [('corpus', CORPUS_PATH), ('colpali', COLPALI_INDEX_DIR), ('clip', CLIP_INDEX_DIR)]:
    exists = '✓' if os.path.exists(path) else '✗'
    print(f'{exists} {name}: {path}')
    if not os.path.exists(path):
        print(f'  ⚠️  Edit the path above to match your inputs')

## Step 4: Clone repo and set up environment

In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient

WORKING_DIR = '/kaggle/working'
REPO_PATH = os.path.join(WORKING_DIR, 'cxr-rag-system')

# Clone repo
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mohamedtaha77/cxr-rag-system.git', REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

sys.path.insert(0, REPO_PATH)

# Load secrets into environment
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
NGROK_TOKEN = secrets.get_secret('NGROK_TOKEN')

print('✓ Repo ready')
print('✓ Secrets loaded')

## Step 5: Load the Gradio app with local indexes

In [ ]:
import importlib.util

# Load the app module
spec = importlib.util.spec_from_file_location(
    'app_gradio', 
    os.path.join(REPO_PATH, 'app', 'app_gradio.py')
)
app_mod = importlib.util.module_from_spec(spec)

# Inject local index paths into module globals BEFORE execution
import importlib, types
import pandas as pd

# Pre-populate module-level vars to skip HF download
app_mod_dict = {
    '__name__': 'app_gradio',
    '__file__': spec.origin,
}

# Execute the module
spec.loader.exec_module(app_mod)

# Override the global vars to use local files instead of downloaded
app_mod.INDEX_DIR = os.path.dirname(COLPALI_INDEX_DIR)
app_mod.path_to_impression = dict(zip(pd.read_csv(CORPUS_PATH)['image_path'], pd.read_csv(CORPUS_PATH)['impression']))

# Override the retriever loaders to use the local paths
def _get_colpali_local():
    if app_mod._colpali is None:
        from src.retrieval.colpali_retriever import ColPaliRetriever
        app_mod._colpali = ColPaliRetriever.from_index(COLPALI_INDEX_DIR)
    return app_mod._colpali

def _get_clip_local():
    if app_mod._clip is None:
        from src.retrieval.clip_retriever import CLIPRetriever
        app_mod._clip = CLIPRetriever()
        app_mod._clip.load_index(CLIP_INDEX_DIR)
    return app_mod._clip

app_mod.get_colpali = _get_colpali_local
app_mod.get_clip = _get_clip_local

print('✓ App loaded with local index paths')

## Step 6: Launch Gradio + create ngrok tunnel

In [ ]:
import threading
import time
from pyngrok import ngrok

# Launch Gradio in background thread
def run_gradio():
    app_mod.demo.launch(server_port=7860, share=False, server_name='0.0.0.0', quiet=True)

gradio_thread = threading.Thread(target=run_gradio, daemon=True)
gradio_thread.start()

print('Starting Gradio... (~20 sec)')
time.sleep(20)

# Setup ngrok
ngrok.kill()  # Kill any existing tunnels
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(7860)

print('\n' + '=' * 60)
print(f'✓ PUBLIC URL: {public_url}')
print('=' * 60)
print('\nOpen this URL in a new browser tab.')
print('First request will be slow (~60s — models loading).')
print('Subsequent requests fast (~5-10s).')
print('\nKeep this notebook running for the URL to stay alive.')

## Step 7 (optional): Stop the app

Run this when done with the demo:

In [ ]:
# Kill ngrok and gradio
ngrok.kill()
app_mod.demo.close()
print('✓ Demo stopped')